## Model Experiments

##### Importing Librares

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import yaml
from src.utils.project_paths import ensure_project_root_on_path, get_project_root, get_config_path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
ensure_project_root_on_path()
project_root = get_project_root()
config_path = get_config_path()
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

##### Loading Data

In [3]:
processed_path = project_root / config["data"]["processed_file"]
df = pd.read_csv(processed_path)

In [4]:
df.shape

(7043, 27)

##### Separate X and y

In [5]:
TARGET = config["model"]["target"]

In [22]:
df_new = df.drop(columns=["City"])
X = df_new.drop(columns=[TARGET])
y = df[TARGET]

In [23]:
y.value_counts(normalize = True)

Churn Value
0    0.73463
1    0.26537
Name: proportion, dtype: float64

##### Train/Test Split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config["model"]["test_size"],
    random_state=config["model"]["random_state"],
    stratify=y
)

In [25]:
y_train.value_counts(
    normalize=True
)

Churn Value
0    0.734647
1    0.265353
Name: proportion, dtype: float64

In [26]:
y_test.value_counts(
    normalize=True
)

Churn Value
0    0.734564
1    0.265436
Name: proportion, dtype: float64

In [27]:
categorical_cols = (

    X_train
    .select_dtypes(
        include="object"
    )
    .columns

)

numerical_cols = (

    X_train
    .select_dtypes(
        exclude="object"
    )
    .columns

)

In [28]:
categorical_cols

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Tenure Group'],
      dtype='object')

In [29]:
numerical_cols

Index(['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV',
       'Total Services', 'Avg Monthly Spend', 'High Value Customer',
       'Monthly Contract'],
      dtype='object')

In [30]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [31]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [32]:
preprocessor = ColumnTransformer(\
    transformers=[
        (
            "numerical",
            numeric_pipeline,
            numerical_cols
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_cols
        )
    ]
)

In [33]:
models = {
    "Logistic Regression":
        LogisticRegression(
            max_iter = 1000,
            class_weight = "balanced"
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators = 200,
            class_weight = "balanced",
            random_state = config["model"]["random_state"]
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators = 300,
            learning_rate = 0.05,
            max_depth = 5,
            eval_metric = "logloss",
            random_state = config["model"]["random_state"]
        )
}

In [34]:
results = []

for name,model in models.items():
    clf = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),

            (
                "model",
                model
            )
        ]
    )

    clf.fit(
        X_train,
        y_train
    )

    y_pred = clf.predict(
        X_test
    )

    y_prob = clf.predict_proba(
        X_test
    )[:,1]

    results.append(
        {
            "Model":name,
            "Accuracy":
            accuracy_score(
                y_test,
                y_pred
            ),

            "Precision":
            precision_score(
                y_test,
                y_pred
            ),

            "Recall":
            recall_score(
                y_test,
                y_pred
            ),

            "F1":
            f1_score(
                y_test,
                y_pred
            ),

            "ROC_AUC":
            roc_auc_score(
                y_test,
                y_prob
            )
        }
    )

In [35]:
results_df = pd.DataFrame(
    results
)

results_df.sort_values(
    by = "ROC_AUC",
    ascending = False
)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.749468,0.518519,0.786096,0.624867,0.852029
2,XGBoost,0.789922,0.623418,0.526738,0.571014,0.846807
1,Random Forest,0.779986,0.572727,0.673797,0.619165,0.840660
